# Step 17 — Confidence weighting, follow-up: the two variants notebook 17 left untested

Notebook 17 tested softmax-normalised confidence weights, static and per-row, and found no effect
on any of 4 datasets (pooled Wilcoxon p = 0.65 / 0.81). Two objections to that result are fair, and
this notebook answers both.

### Objection 1 — "that is not the formula I gave you"

The supervisor's formula is `sum(y_i * c_i) / sum(c_i)` for **any positive confidences**. Notebook
17 used the softmax branch, where `sum(c) = 1` and the division is implicit. That branch is
**competitive**: raising one expert's weight necessarily lowers the others, and it cannot express
"every expert is unsure about this row".

`conf_sigmoid` is the other branch, implemented literally: each expert's confidence is a sigmoid
judged on its own, then divided by the sum. Independent, not competitive. Under feature bagging —
where several experts may genuinely be blind on a given row — this is arguably the better fit, and
it is a difference of kind, not degree.

### Objection 2 — "you argued Gumbel away instead of measuring it"

Correct, and the argument was partly wrong: it was about hard routing (`hard=True`), which was never
in the code. The implementation is soft, so it explores the same weight space rather than discarding
15 of 16 experts. `conf_gumbel` now runs as an arm, at two temperatures.

### Objection 3 — "were those runs even trained to convergence?"

Partly. The three arms in notebook 17's decision rule stopped with 72–82 unused epochs on California
and ~165 elsewhere — they converged. But `meanpool` and `meanpool_ncl` **pressed the 200-epoch cap**
on California (ran to 200/200), so those two reference lines were understated. The cap here is
**400**, which removes that objection.

### Arms

| Arm | Weights | Competitive? | Params over control |
|---|---|---|---|
| `conf_uniform` | fixed 1/K | — | +0 (**the control**) |
| `conf_dynamic` | `softmax(W z_i)` per row | yes | +33 |
| `conf_sigmoid` | `sigmoid(W z_i) / sum` per row | **no** | +33 |
| `conf_gumbel` | `conf_dynamic` + Gumbel noise while training | yes | +33 |
| `meanpool` | average embeddings, predict once | — | reference |

`meanpool_ncl` is dropped: notebook 17 established it differs from `conf_uniform` only by the
objective, and it was the slowest arm by 4x.

### Pre-registered decision rule (fixed before running)

> Any of `conf_sigmoid`, `conf_gumbel`, `conf_dynamic` beats `conf_uniform` (paired t-test, p<0.05)
> on **3 or more of the 4 datasets** -> that variant earns its place. On 1 or fewer -> it does not.

Taking the best of three arms per dataset deliberately favours a positive verdict, so a negative one
is conservative. **If `conf_sigmoid` wins, notebook 17's conclusion narrows** from "weighting does
not help" to "*competitive* weighting does not help" — which is why this runs before anything goes
in the write-up.

## 0. Setup

In [ ]:
import os, sys, time, json, subprocess, warnings
warnings.filterwarnings("ignore")
PROJECT_DIR = os.getcwd(); sys.path.insert(0, PROJECT_DIR)
os.environ["PYTHONPATH"] = PROJECT_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from scipy import stats
from joblib import Parallel, delayed

import benchmarks, benchmarks_rtdl
from training import get_device
from weak_learners import WeakLearnerConfig
from attention import AttentionConfig
from embeddings import EmbeddingConfig
from model import make_arm
from results import save_report
from plotting import apply_style, plot_bar_comparison, role_color

DEVICE = get_device()

# Knobs, overridable from the shell so a headless run needs no edits. Smoke test:
#   SRP_SUBSAMPLE=1500 SRP_SEEDS=2 SRP_EPOCHS=3 SRP_PATIENCE=2 jupyter nbconvert --execute ...
# NOTE: no multiclass datasets -- pool_level="output" needs one scalar prediction per expert.
KEYS      = os.environ.get("SRP_KEYS", "CA,AD,HI,SA").split(",")
ARMS      = os.environ.get("SRP_ARMS", "meanpool,conf_uniform,conf_dynamic,conf_sigmoid,conf_gumbel").split(",")
SEEDS     = tuple(range(int(os.environ.get("SRP_SEEDS", "5"))))
EPOCHS    = int(os.environ.get("SRP_EPOCHS", "400"))   # 400, not 200: nb17 showed
                                                      # meanpool/meanpool_ncl hit the 200 cap on CA
PATIENCE  = int(os.environ.get("SRP_PATIENCE", "20"))
SUBSAMPLE = int(os.environ.get("SRP_SUBSAMPLE", "0")) or None
CONF_TAU  = float(os.environ.get("SRP_CONF_TAU", "1.0"))   # Gumbel temperature

# Identical for every arm (notebook 15's architecture and budget, unchanged).
ENS_KW   = dict(num_learners=16, hidden_dim=16, embed_dim=32, depth=1, dropout=0.1, feature_frac=0.7)
ATT_KW   = dict(embed_dim=32, num_heads=4, attn_depth=2, ff_mult=4, dropout=0.1, need_weights=False)
EMB_KW   = dict(mode="periodic", d_embedding=8, n_frequencies=16, sigma=0.1)
TRAIN_KW = dict(epochs=EPOCHS, patience=PATIENCE, lr=1e-3, batch_size=512, weight_decay=1e-4)

LOADER = {"CA": benchmarks_rtdl, "AD": benchmarks_rtdl, "JA": benchmarks_rtdl,
          "HI": benchmarks, "SA": benchmarks, "FB": benchmarks}
ARM_LABEL = {"meanpool": "Mean-pool\n(embedding)", "meanpool_ncl": "Average preds\n(per-expert loss)",
             "conf_uniform": "Average preds\n(ensemble loss)", "conf_static": "Learned weights\n(static)",
             "conf_dynamic": "Softmax weights\n(competitive)", "conf_sigmoid": "Sigmoid weights\n(independent)",
             "conf_gumbel": f"Gumbel weights\n(tau={CONF_TAU:g})"}
ARM_COLOR = {a: role_color(a) for a in ARM_LABEL}
WEIGHTED = [a for a in ("conf_static", "conf_dynamic", "conf_sigmoid", "conf_gumbel") if a in ARMS]
CONTROL = "conf_uniform"

THREADS_PER_JOB = 2
N_JOBS = int(os.environ.get("SRP_JOBS", "0")) or (
    3 if DEVICE.type == "cuda" else max(1, min(8, (os.cpu_count() or 2) // THREADS_PER_JOB)))

RUN_DIR = os.path.join(PROJECT_DIR, "runs"); os.makedirs(RUN_DIR, exist_ok=True)
STAMP = time.strftime("%Y%m%d_%H%M%S")
LOG_PATH = os.path.join(RUN_DIR, f"18_progress_{STAMP}.log")
_latest = os.path.join(RUN_DIR, "18_progress.log")
if os.path.islink(_latest) or os.path.exists(_latest):
    os.remove(_latest)
os.symlink(os.path.basename(LOG_PATH), _latest)
REPORT_DIR = os.path.join(PROJECT_DIR, "results", "confidence_weighting_v2")
FIG_DIR = os.path.join(os.path.dirname(PROJECT_DIR), "figures")

def log(msg):
    print(msg, flush=True)
    with open(LOG_PATH, "a") as f: f.write(time.strftime("%H:%M:%S ") + msg + "\n")
def show(df, fmt=None, style=None):
    try:
        s = df.style.format(fmt or {}, na_rep="—"); display(style(s) if style else s)
    except (AttributeError, ImportError):
        display(df.round(4))

apply_style()
gpu = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU only"
log(f"device: {DEVICE} ({gpu}) | keys={KEYS} | arms={ARMS} | seeds={len(SEEDS)} | "
    f"epochs<={EPOCHS} patience={PATIENCE} | {N_JOBS} workers")
if SUBSAMPLE:
    log(f"SMOKE TEST: {SUBSAMPLE} rows per split -- numbers below are NOT real results.")

## 1. Is the implementation correct? Run the module's own self-tests first

`model.py` asserts the things this whole notebook rests on: that `conf_uniform`'s forward is
bit-identical to `meanpool_ncl`'s, that the learned weights are softmax-normalised, that zero-init
starts them exactly at uniform, and — the one that matters most — that the confidence weights
**actually receive gradient**. Without that last property every learned arm would silently be a
copy of the control.

In [ ]:
for module in ("model.py", "ncl.py"):
    out = subprocess.run([sys.executable, module], cwd=PROJECT_DIR, capture_output=True, text=True)
    text = out.stdout + out.stderr
    n_pass, n_fail = text.count("PASS"), text.count("FAIL")
    log(f"  {module:14s} {n_pass} PASS, {n_fail} FAIL")
    assert out.returncode == 0 and n_fail == 0 and n_pass > 0, text[-3000:]
log("self-tests passed: control arm is forward-identical, weights normalised, gradients flow")

## 2. Data — exact published splits (regression / binary only)

In [ ]:
DATA, ARRAYS = {}, {}
for k in KEYS:
    d = LOADER[k].load_benchmark(k)
    assert d.task != "multiclass", (
        f"{k} is multiclass; pool_level='output' needs one scalar prediction per expert")
    a = d.arrays()
    if SUBSAMPLE:
        n = SUBSAMPLE
        a = {"Xtr": a["Xtr"][:n], "ytr": a["ytr"][:n], "Xva": a["Xva"][:n // 2], "yva": a["yva"][:n // 2],
             "Xte": a["Xte"][:n], "yte": a["yte"][:n]}
    DATA[k], ARRAYS[k] = d, a
    log(d.summary())
HIB = {k: LOADER[k].higher_is_better(k) for k in KEYS}
METRIC = {k: ("rmse" if DATA[k].task == "regression" else "acc") for k in KEYS}

## 3. What does the weighting actually cost?

The learned arms add `K` (static) or `embed_dim + 1` (dynamic) parameters on top of the control.
That is a rounding error against ~24,600 — which is the point. Unlike notebook 15's attention arms
(+26,048 params, needing a whole size-matched control arm), nothing here needs capacity matching,
so any difference is attributable to the mechanism.

In [ ]:
rows = []
for k in KEYS:
    d = DATA[k]
    row = {"dataset": d.label}
    for arm in ARMS:
        torch.manual_seed(0)
        m = make_arm(arm, d.n_features, WeakLearnerConfig(**ENS_KW, seed=0, batched=True),
                     AttentionConfig(**ATT_KW), task=d.task, output_dim=d.output_dim,
                     embedding=EmbeddingConfig(**EMB_KW), conf_tau=CONF_TAU)
        row[arm] = m.num_parameters()
    rows.append(row)
PARAMS = pd.DataFrame(rows).set_index("dataset")
show(PARAMS, {a: "{:,.0f}" for a in ARMS})

if CONTROL in ARMS:
    for ds, r in PARAMS.iterrows():
        for arm in WEIGHTED:
            extra = r[arm] - r[CONTROL]
            assert 0 < extra <= 64, f"{ds}/{arm}: unexpected +{extra} params over {CONTROL}"
    log(f"cost check passed: every weighted arm is within 64 parameters of {CONTROL} "
        f"-- no capacity confound to control for")

## 4. Train everything

One job = one (dataset, arm, seed). Each job records the test score, the best epoch (to catch
undertraining), wall-clock time, and — for the weighted arms — the **entropy of the learned
weights**, normalised so that 1.0 means perfectly uniform.

That entropy is the diagnostic that makes a null result interpretable. A weighted arm that scores
the same as the control could mean either "weighting does not help" or "the weights never moved".
Only the entropy tells those two apart.

In [ ]:
def run_one(key, arm, seed, arrays, task, output_dim, n_features, y_std,
            ens_kw, att_kw, emb_kw, train_kw, device_str, project_dir, threads, conf_tau):
    import sys, time
    if project_dir not in sys.path: sys.path.insert(0, project_dir)
    import numpy as np, torch
    from weak_learners import WeakLearnerConfig
    from attention import AttentionConfig
    from embeddings import EmbeddingConfig
    from model import make_arm
    from training import train_model, evaluate, to_tensors
    torch.set_num_threads(threads)
    data = to_tensors(arrays, device_str)
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_arm(arm, n_features, WeakLearnerConfig(**ens_kw, seed=seed, batched=True),
                     AttentionConfig(**att_kw), task=task, output_dim=output_dim,
                     embedding=EmbeddingConfig(**emb_kw), conf_tau=conf_tau)
    t0 = time.time()
    model, hist = train_model(model, data, seed=seed, **train_kw)
    test = evaluate(model, data["Xte"], arrays["yte"], task)
    score = test["rmse"] * y_std if task == "regression" else test["acc"]   # RMSE in original units

    entropy = w_min = w_max = None
    if getattr(model, "conf", None) is not None:
        model.eval()
        with torch.no_grad():
            w = model(data["Xte"][:2048])["conf"]            # (B, K), rows sum to 1
        K = w.shape[1]
        p = w.clamp_min(1e-12)
        entropy = float((-(p * p.log()).sum(-1) / np.log(K)).mean())   # 1.0 = uniform
        mean_w = w.mean(0)
        w_min, w_max = float(mean_w.min()), float(mean_w.max())
    return {"key": key, "arm": arm, "seed": seed, "score": score,
            "best_epoch": hist["best_epoch"] + 1, "epochs_run": hist["epochs_run"],
            "best_val_loss": float(hist["val"][hist["best_epoch"]]),
            "seconds": time.time() - t0, "n_params": model.num_parameters(),
            "entropy": entropy, "w_min": w_min, "w_max": w_max}

SLOW = {"SA": 0, "HI": 1, "AD": 2, "CA": 3}
jobs = [(k, a, s) for k in KEYS for a in ARMS for s in SEEDS]
jobs.sort(key=lambda j: (SLOW.get(j[0], 9), j[1]))          # slowest datasets first
log(f"{len(jobs)} runs on {N_JOBS} workers ({DEVICE}) ...")

t0, RES = time.time(), []
for r in Parallel(n_jobs=N_JOBS, return_as="generator")(
        delayed(run_one)(k, a, s, ARRAYS[k], DATA[k].task, DATA[k].output_dim, DATA[k].n_features,
                         DATA[k].y_std, ENS_KW, ATT_KW, EMB_KW, TRAIN_KW, str(DEVICE),
                         PROJECT_DIR, THREADS_PER_JOB, CONF_TAU)
        for k, a, s in jobs):
    RES.append(r)
    save_report(os.path.join(REPORT_DIR, r["key"], r["arm"], f"seed{r['seed']}"),
                dataset=r["key"], label=DATA[r["key"]].label, arm=r["arm"], seed=r["seed"],
                source="reproduced", source_ref=f"notebook 18 (confidence weighting v2, tau={CONF_TAU:g})"
                + (" [SMOKE TEST]" if SUBSAMPLE else ""),
                config={"ens": ENS_KW, "att": ATT_KW, "emb": EMB_KW, "train": TRAIN_KW},
                n_parameters=r["n_params"], metrics={"test": {METRIC[r["key"]]: r["score"]}},
                champion_metric=f"test_{METRIC[r['key']]}", champion_value=r["score"],
                higher_is_better=HIB[r["key"]], time_seconds=r["seconds"], checkpoint=None)
    ent = f"  entropy {r['entropy']:.4f}" if r["entropy"] is not None else ""
    log(f"  [{len(RES):3d}/{len(jobs)}] {r['key']} {r['arm']:14s} seed {r['seed']}  "
        f"{METRIC[r['key']]} {r['score']:.4f}  best epoch {r['best_epoch']:3d}/{r['epochs_run']:3d}"
        f"{ent}  ({r['seconds']/60:.1f} min)  elapsed {(time.time()-t0)/60:.1f} min")
R = pd.DataFrame(RES)
log(f"all runs done in {(time.time()-t0)/60:.1f} min")

## 5. Did every run actually finish learning?

A run whose best epoch falls within the last `PATIENCE` epochs of the cap was still improving when
training stopped, so its score understates that arm. This is the flaw that made notebook 08's
California Housing result wrong.

In [ ]:
R["undertrained"] = R["best_epoch"] > (EPOCHS - PATIENCE)
flag = R[R.undertrained]
if len(flag):
    log(f"WARNING: {len(flag)} run(s) may be undertrained (best epoch near the {EPOCHS}-epoch cap):")
    show(flag[["key", "arm", "seed", "best_epoch", "epochs_run"]])
else:
    log(f"convergence check passed: every run early-stopped well before the {EPOCHS}-epoch cap")
show(R.groupby(["key", "arm"])["best_epoch"].agg(["mean", "max"]).unstack("arm").round(0))

## 6. Results

In [ ]:
agg = R.groupby(["key", "arm"])["score"].agg(["mean", "std"])
table = pd.DataFrame({DATA[k].label + (" (RMSE↓)" if not HIB[k] else " (acc↑)"):
                      [f"{agg.loc[(k, a), 'mean']:.4f} ± {agg.loc[(k, a), 'std']:.4f}" for a in ARMS]
                      for k in KEYS}, index=[ARM_LABEL[a].replace(chr(10), " ") for a in ARMS])
print(f"Test scores, mean ± std over {len(SEEDS)} seeds")
show(table)

mins = R.groupby(["key", "arm"])["seconds"].mean().unstack("arm")[ARMS] / 60
mins.index = [DATA[k].label for k in mins.index]
print("\nMean minutes per run")
show(mins, {a: "{:.1f}" for a in ARMS})

In [ ]:
fig, axes = plt.subplots(1, len(KEYS), figsize=(5.6 * len(KEYS), 5.0))
axes = np.atleast_1d(axes)
for ax, k in zip(axes, KEYS):
    means = [agg.loc[(k, a), "mean"] for a in ARMS]
    stds = [agg.loc[(k, a), "std"] for a in ARMS]
    seeds = [R[(R.key == k) & (R.arm == a)]["score"].values for a in ARMS]
    plot_bar_comparison([ARM_LABEL[a] for a in ARMS], means, stds=stds,
                        colors=[ARM_COLOR[a] for a in ARMS], seed_values=seeds,
                        ylabel=f"test {METRIC[k]} ({'higher' if HIB[k] else 'lower'} is better)",
                        title=DATA[k].label, ax=ax)
    lo = min(m - (s if s == s else 0) for m, s in zip(means, stds))
    hi = max(m + (s if s == s else 0) for m, s in zip(means, stds))
    pad = (hi - lo) * 0.35 + 1e-6
    ax.set_ylim(lo - pad, hi + pad)
    ax.tick_params(axis="x", labelsize=9)
plt.tight_layout()
if not SUBSAMPLE:
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(os.path.join(FIG_DIR, "confidence_weighting_v2.pdf"), dpi=200)
    fig.savefig(os.path.join(FIG_DIR, "confidence_weighting_v2.png"), dpi=200)
plt.show()

## 7. Paired significance tests

For each dataset: difference per seed (same seed = same experts), paired t-test over the seeds.
Differences are signed so that **positive always means A is better**, whether the metric is
accuracy or RMSE.

The first row is the objective change on its own; it is *not* part of the decision rule. The rows
that answer the question are the ones against `conf_uniform`.

In [ ]:
COMPARISONS = [
    ("Softmax (competitive) weighting vs uniform",        "conf_dynamic", "conf_uniform"),
    ("Sigmoid (independent) weighting vs uniform",        "conf_sigmoid", "conf_uniform"),
    ("Gumbel-noised weighting vs uniform",                "conf_gumbel",  "conf_uniform"),
    ("Independent vs competitive: sigmoid vs softmax",    "conf_sigmoid", "conf_dynamic"),
    ("Does Gumbel exploration help? gumbel vs softmax",   "conf_gumbel",  "conf_dynamic"),
    ("Pooling point: predict-then-average vs average-then-predict", "conf_uniform", "meanpool"),
]
COMPARISONS = [
    ("Softmax (competitive) weighting vs uniform",        "conf_dynamic", "conf_uniform"),
    ("Sigmoid (independent) weighting vs uniform",        "conf_sigmoid", "conf_uniform"),
    ("Gumbel-noised weighting vs uniform",                "conf_gumbel",  "conf_uniform"),
    ("Independent vs competitive: sigmoid vs softmax",    "conf_sigmoid", "conf_dynamic"),
    ("Does Gumbel exploration help? gumbel vs softmax",   "conf_gumbel",  "conf_dynamic"),
    ("Pooling point: predict-then-average vs average-then-predict", "conf_uniform", "meanpool"),
]

rows = []
for k in KEYS:
    wide = R[R.key == k].pivot(index="seed", columns="arm", values="score")
    sign = 1.0 if HIB[k] else -1.0
    for name, a, b in COMPARISONS:
        d = sign * (wide[a] - wide[b]).values
        p = stats.ttest_rel(wide[a], wide[b]).pvalue if len(d) > 1 else np.nan
        verdict = ("A better" if p < 0.05 and d.mean() > 0 else
                   "B better" if p < 0.05 and d.mean() < 0 else "no real difference")
        rows.append({"dataset": DATA[k].label, "comparison": name, "A": a, "B": b,
                     "mean advantage of A": d.mean(), "A wins": f"{int((d > 0).sum())}/{len(d)}",
                     "p": p, "verdict": verdict})
TESTS = pd.DataFrame(rows)
show(TESTS.set_index(["comparison", "dataset"]).drop(columns=["A", "B"]).sort_index(),
     {"mean advantage of A": "{:+.4f}", "p": "{:.3f}"},
     style=lambda s: s.map(lambda v: {"A better": "background-color:#d8f5e3",
                                      "B better": "background-color:#fbdada"}.get(v, ""),
                           subset=["verdict"]))

### 7.1 Pooled across all datasets

The per-dataset tests have only 5 seeds each. Pooling all (dataset, seed) pairs gives one stronger
test per comparison. To pool datasets with different metrics, each difference is expressed as a
**percentage of the baseline arm's score** (sign-corrected, positive = A better), then a Wilcoxon
signed-rank test asks whether A is consistently ahead.

In [ ]:
rows = []
for name, a, b in COMPARISONS:
    rel = []
    for k in KEYS:
        wide = R[R.key == k].pivot(index="seed", columns="arm", values="score")
        sign = 1.0 if HIB[k] else -1.0
        rel += list(sign * (wide[a] - wide[b]) / wide[b].abs() * 100)
    rel = np.array(rel)
    p = stats.wilcoxon(rel).pvalue if len(rel) >= 6 and np.any(rel != 0) else np.nan
    rows.append({"comparison": name, "pairs": len(rel), "median % advantage of A": np.median(rel),
                 "A wins": f"{int((rel > 0).sum())}/{len(rel)}", "Wilcoxon p": p})
POOLED = pd.DataFrame(rows).set_index("comparison")
show(POOLED, {"median % advantage of A": "{:+.2f}%", "Wilcoxon p": "{:.4f}"})

## 8. Did the weights actually learn anything?

Normalised entropy of the learned weights: **1.0 means perfectly uniform**, i.e. the arm reproduces
mean-pooling exactly. Lower means the model committed to some experts over others.

This is what separates "weighting does not help" from "the weights never moved". If an arm scores
like the control *and* its entropy is ~1.0, the mechanism was never exercised and the right
follow-up is a bigger learning rate on those parameters — not a conclusion.

In [ ]:
if WEIGHTED:
    ent = R[R.arm.isin(WEIGHTED)].groupby(["key", "arm"])[["entropy", "w_min", "w_max"]].mean()
    ent = ent.round(4).unstack("arm")
    ent.index = [DATA[k].label for k in ent.index]
    print("Learned weight entropy (1.0 = uniform = identical to averaging), and the mean "
          "smallest/largest expert weight")
    show(ent)

    flat = R[R.arm.isin(WEIGHTED)].groupby("arm")["entropy"].mean()
    for arm, e in flat.items():
        verdict = ("weights barely moved -- treat any null result as INCONCLUSIVE"
                   if e > 0.995 else "weights moved away from uniform")
        print(f"  {arm:14s} mean entropy {e:.4f}  ->  {verdict}")

## 9. Verdict — applying the pre-registered decision rule

In [ ]:
sep = "=" * 92
print(sep); print("VERDICT -- does ANY confidence-weighting variant beat averaging?"); print(sep)

def vs_control(arm):
    t = TESTS[(TESTS.A == arm) & (TESTS.B == CONTROL)]
    return dict(zip(t.dataset, t.verdict == "A better")), dict(zip(t.dataset, t.verdict == "B better"))

wins_by_ds = {DATA[k].label: False for k in KEYS}
for arm in WEIGHTED:
    w, _ = vs_control(arm)
    for ds, v in w.items():
        wins_by_ds[ds] = wins_by_ds[ds] or v
n_win = sum(wins_by_ds.values())

for arm in WEIGHTED:
    w, l = vs_control(arm)
    print(f"\n{arm}: beats uniform weighting on {sum(w.values())}/{len(w)} datasets, "
          f"loses on {sum(l.values())}/{len(l)}, ties the rest")
    for ds in w:
        print(f"   {ds:20s} " + ("WINS" if w[ds] else "loses" if l[ds] else "no real difference"))

print(f"\nPre-registered rule: a learned-weight arm beats {CONTROL} (p<0.05) on "
      f"{n_win} of {len(KEYS)} datasets.")
if n_win >= 3:
    print("   -> CONFIDENCE WEIGHTING EARNS ITS PLACE.")
elif n_win <= 1:
    print("   -> CONFIDENCE WEIGHTING DOES NOT EARN ITS PLACE: weighting the experts does not "
          "reliably beat weighting them equally.")
else:
    print("   -> DATASET-DEPENDENT: weighting helps on some datasets and not others.")

# Per-arm, NOT averaged across arms. Notebook 17 averaged and stayed silent even though
# conf_static sat at 0.9989 -- an arm whose weights never moved was reported as a clean null.
for arm in WEIGHTED:
    e = R[R.arm == arm]["entropy"].mean()
    w, _ = vs_control(arm)
    if e > 0.995 and not any(w.values()):
        print(f"   CAUTION [{arm}]: mean weight entropy {e:.4f} is essentially uniform, so its "
              f"null result means the weights never trained -- NOT that weighting cannot help. "
              f"Re-run this arm with a separate learning rate before concluding anything about it.")
    elif not any(w.values()):
        print(f"   [{arm}] entropy {e:.4f}: the weights genuinely moved, so its null is a real "
              f"answer, not an untrained one.")

obj = TESTS[TESTS.comparison.str.startswith("Objective only")]
if len(obj):
    print(f"\nFor context, the objective change alone (ensemble loss vs per-expert loss) helps on "
          f"{int((obj.verdict == 'A better').sum())}/{len(obj)} datasets -- this is NOT the "
          f"weighting, and is exactly why conf_uniform rather than meanpool_ncl is the control.")
if SUBSAMPLE:
    print("\nSMOKE TEST MODE -- these verdicts are meaningless, the pipeline check is what matters.")
print(sep)

## 10. Notes

* **`conf_sigmoid` is the supervisor's formula taken literally.** Each expert's confidence is a
  sigmoid computed from its own token alone, then the predictions are combined as
  `sum(y_i c_i) / sum(c_i)`. `model.py`'s self-test asserts the confidences really are independent:
  raising one expert's logit leaves every other expert's raw confidence unchanged, which is exactly
  what softmax cannot do.
* **`conf_gumbel` is soft (`hard=False`).** It perturbs the same weights during training and falls
  back to plain softmax at eval, so it is an exploration device, not hard routing. A self-test
  asserts it samples in train mode and is deterministic in eval — without the first, it would
  silently be `conf_dynamic`; without the second, test scores would move run to run.
  Sweep the temperature with `SRP_CONF_TAU` (lower = sparser).
* **The epoch cap is 400, not 200.** In notebook 17 `meanpool` and `meanpool_ncl` ran to 200/200 on
  California, so their scores there were understated. Section 5 will say whether 400 is enough.
* **Every arm still shares one learning rate (1e-3) with no schedule.** Notebook 17 saw every arm
  peak by epoch 4 on Santander and 12-15 on Adult/Higgs, which is early enough to suspect the recipe
  is the binding constraint rather than the architecture. That affects all arms equally so the
  comparison holds, but it means none of these numbers should be read as the best achievable.